# Data Preprocessing (cleaning, identify gaps, and standardization)

**Contributions: Shaotai Hu,**

**documentation link https://docs.google.com/document/d/1vURMI1VaFVwlVZkdtHFTEj1ys6Bwc0y1Oz1i5GJ8c_Q/edit?usp=sharing**

In [1]:
# data cleaning, identify gaps, and standardization - Shaotai Hu
# feel free to add more
# import raw data

import pandas as pd
import numpy as np

ep_base = pd.read_csv("/content/Employer Base Dataset (Spring 2020 - Summer 2025).csv")
ep_event = pd.read_csv("/content/Employer Event Dataset (Spring 2020 - Summer 2025).csv")
ep_job = pd.read_csv("/content/Employer Job Dataset (Spring 2020 - Summer 2025).csv")

ep_base.head(5)

,Unnamed: 0,Name,ID,Name.1,Employer Size,Created At Date,Email,Website,Count
0,1,"Lawhon & Associates, Inc.",392601,Environmental Services,10 - 50,2019-02-05,kblitzer@lawhon-assoc.com,http://www.lawhon-assoc.com,1
1,2,Modulus Global Inc.,385582,Management Consulting,1 - 10,2019-01-18,jeff@modulus.io,https://www.modulusglobal.com/,1
2,3,Chameleon Coffee,1008255,Food & Beverage,100 - 250,2025-01-13,careers@chameleoncoffee.com,https://www.chameleoncoffee.com,1
3,4,Brookstone Management Concepts,422901,"Advertising, PR & Marketing",50 - 100,2019-04-18,hr@brookstonemgmtconcepts.com,https://brookstonemgmtconcepts.com/,1
4,5,Department 13,252845,Electronic & Computer Hardware,10 - 50,2018-02-21,hr@department13.com,http://department13.com,1


In [2]:
ep_event.head(5)

,Events Name,Events Start Date Date,Events ID,Employers Name,Employers ID,Event Medium Name,Student Attendees Count
0,Chat & Connect: Explore Behavior Technician Jo...,2025-08-27,1801479,Autism Learning Partners,72869,Virtual,0
1,Sevita Virtual Hiring Event- Connect. Explore....,2025-08-26,1796898,Sevita,59790,Hybrid,0
2,Go Beyond the Classroom: Educational Consultin...,2025-08-29,1794932,Cardinal Education,17654,Virtual,1
3,Amazon 2025 Warehouse Virtual Information Sess...,2025-08-21,1793355,Amazon,22003,Virtual,3
4,Explore Your Future with Viridien: Student Inf...,2025-08-19,1792463,Viridien,35449,Virtual,6


In [3]:
ep_job.head(5)

,Employer Name,Employer ID,Job Salaries Min Salary In Dollars,Job Salaries Max Salary In Dollars,Jobs Created At Date,Jobs Title,Jobs ID,Applications Count
0,AlumBoost,1061634,50,50,2025-08-30,Pre-Med and Pre-Law Tutor (Remote),10231126,0
1,Natural Masters By New Beginnings,1061534,NaN,NaN,2025-08-29,Marketing Coordinator,10228882,1
2,Natural Masters By New Beginnings,1061534,NaN,NaN,2025-08-29,Sales Associate,10228857,0
3,Sarpel Household,1061467,30,40,2025-08-29,"Morning Routine Coach (10+ hours), Cambridge, MA",10227979,0
4,Seven Muses Media LLC,1061309,20,30,2025-08-28,Crowdfunding assistance for record label / boo...,10221579,1


In [4]:
# understanding the shape and columns in the dataset
print("ep_base shape: ", ep_base.shape)
print("ep_event shape: ", ep_event.shape)
print("ep_job shape: ", ep_job.shape)

print(ep_base.columns)
print(ep_event.columns)
print(ep_job.columns)

ep_base shape:  (56089, 9)
ep_event shape:  (12318, 7)
ep_job shape:  (886840, 8)
Index(['Unnamed: 0', 'Name', 'ID', 'Name.1', 'Employer Size',
       'Created At Date', 'Email', 'Website', 'Count'],
      dtype='object')
Index(['Events Name', 'Events Start Date Date', 'Events ID', 'Employers Name',
       'Employers ID', 'Event Medium Name', 'Student Attendees Count'],
      dtype='object')
Index(['Employer Name', 'Employer ID', 'Job Salaries Min Salary In Dollars',
       'Job Salaries Max Salary In Dollars', 'Jobs Created At Date',
       'Jobs Title', 'Jobs ID', 'Applications Count'],
      dtype='object')


In [5]:
# general cleaning

# standardize column names
ep_base.columns = ep_base.columns.str.strip().str.lower().str.replace(" ", "_")
ep_event.columns = ep_event.columns.str.strip().str.lower().str.replace(" ", "_")
ep_job.columns = ep_job.columns.str.strip().str.lower().str.replace(" ", "_")

# matching the primary key/column name
ep_base = ep_base.rename(columns={"id": "employer_id"})
ep_event = ep_event.rename(columns={"employers_id": "employer_id"})
ep_job = ep_job.rename(columns={"employer_id": "employer_id"})

# lower case everything in the datasets
ep_base = ep_base.map(lambda x: x.lower() if isinstance(x, str) else x)
ep_event = ep_event.map(lambda x: x.lower() if isinstance(x, str) else x)
ep_job = ep_job.map(lambda x: x.lower() if isinstance(x, str) else x)

# drop fully empty rows
ep_base = ep_base.replace(r"^\s*$", np.nan, regex=True).dropna(how="all")
ep_event = ep_event.replace(r"^\s*$", np.nan, regex=True).dropna(how="all")
ep_job = ep_job.replace(r"^\s*$", np.nan, regex=True).dropna(how="all")

ep_job.head(5)

,employer_name,employer_id,job_salaries_min_salary_in_dollars,job_salaries_max_salary_in_dollars,jobs_created_at_date,jobs_title,jobs_id,applications_count
0,alumboost,1061634,50,50,2025-08-30,pre-med and pre-law tutor (remote),10231126,0
1,natural masters by new beginnings,1061534,NaN,NaN,2025-08-29,marketing coordinator,10228882,1
2,natural masters by new beginnings,1061534,NaN,NaN,2025-08-29,sales associate,10228857,0
3,sarpel household,1061467,30,40,2025-08-29,"morning routine coach (10+ hours), cambridge, ma",10227979,0
4,seven muses media llc,1061309,20,30,2025-08-28,crowdfunding assistance for record label / boo...,10221579,1


In [6]:
# employer base cleaning

# drop duplicates
ep_base = ep_base.drop_duplicates(subset="employer_id")

# dropping meaningless columns, count is just all 1
ep_base["count"].unique()
ep_base = ep_base.drop(columns=["unnamed:_0", "count"])

# rename to employer type
ep_base = ep_base.rename(columns={"name.1": "employer_type"})

# splitting employer_size to min and max columns
size_split = ep_base["employer_size"].str.split(" - ", expand=True)
ep_base["employer_size_min"] = pd.to_numeric(size_split[0], errors="coerce")
ep_base["employer_size_max"] = pd.to_numeric(size_split[1], errors="coerce")
ep_base = ep_base.drop(columns=["employer_size"])

# date reformat
ep_base["created_at_date"] = pd.to_datetime(ep_base["created_at_date"], errors="coerce")
ep_base = ep_base.rename(columns={"created_at_date": "employer_creation_date"})

ep_base.head(5)

,name,employer_id,employer_type,employer_creation_date,email,website,employer_size_min,employer_size_max
0,"lawhon & associates, inc.",392601,environmental services,2019-02-05,kblitzer@lawhon-assoc.com,http://www.lawhon-assoc.com,10.0,50.0
1,modulus global inc.,385582,management consulting,2019-01-18,jeff@modulus.io,https://www.modulusglobal.com/,1.0,10.0
2,chameleon coffee,1008255,food & beverage,2025-01-13,careers@chameleoncoffee.com,https://www.chameleoncoffee.com,100.0,250.0
3,brookstone management concepts,422901,"advertising, pr & marketing",2019-04-18,hr@brookstonemgmtconcepts.com,https://brookstonemgmtconcepts.com/,50.0,100.0
4,department 13,252845,electronic & computer hardware,2018-02-21,hr@department13.com,http://department13.com,10.0,50.0


In [7]:
# employer event cleaning

# drop duplicates
ep_event = ep_event.drop_duplicates(subset="events_id")

# student count numeric type
ep_event["student_attendees_count"] = pd.to_numeric(ep_event["student_attendees_count"], errors="coerce")

# rename date and medium name
ep_event = ep_event.rename(columns={"events_start_date_date": "event_date"})
ep_event = ep_event.rename(columns={"event_medium_name": "event_medium"})

# date reformat
ep_event["event_date"] = pd.to_datetime(ep_event["event_date"], errors="coerce")

ep_event.head(5)

,events_name,event_date,events_id,employers_name,employer_id,event_medium,student_attendees_count
0,chat & connect: explore behavior technician jo...,2025-08-27,1801479,autism learning partners,72869,virtual,0
1,sevita virtual hiring event- connect. explore....,2025-08-26,1796898,sevita,59790,hybrid,0
2,go beyond the classroom: educational consultin...,2025-08-29,1794932,cardinal education,17654,virtual,1
3,amazon 2025 warehouse virtual information sess...,2025-08-21,1793355,amazon,22003,virtual,3
4,explore your future with viridien: student inf...,2025-08-19,1792463,viridien,35449,virtual,6


In [8]:
# employer job cleaning

# drop duplicates
ep_job = ep_job.drop_duplicates(subset="jobs_id")

# renames date, min and max salary columns
ep_job = ep_job.rename(columns={"jobs_created_at_date": "job_creation_date"})
ep_job = ep_job.rename(columns={"job_salaries_min_salary_in_dollars": "min_salary"})
ep_job = ep_job.rename(columns={"job_salaries_max_salary_in_dollars": "max_salary"})

#salary datatype to float
ep_job["min_salary"] = ep_job["min_salary"].str.replace(",", "", regex=False)
ep_job["min_salary"] = ep_job["min_salary"].astype(float).fillna(0.0)

ep_job["max_salary"] = ep_job["max_salary"].str.replace(",", "", regex=False)
ep_job["max_salary"] = ep_job["max_salary"].astype(float).fillna(0.0)

# date reformat
ep_job["job_creation_date"] = pd.to_datetime(ep_job["job_creation_date"], errors="coerce")

ep_job.head(5)


,employer_name,employer_id,min_salary,max_salary,job_creation_date,jobs_title,jobs_id,applications_count
0,alumboost,1061634,50.0,50.0,2025-08-30,pre-med and pre-law tutor (remote),10231126,0
1,natural masters by new beginnings,1061534,0.0,0.0,2025-08-29,marketing coordinator,10228882,1
2,natural masters by new beginnings,1061534,0.0,0.0,2025-08-29,sales associate,10228857,0
3,sarpel household,1061467,30.0,40.0,2025-08-29,"morning routine coach (10+ hours), cambridge, ma",10227979,0
4,seven muses media llc,1061309,20.0,30.0,2025-08-28,crowdfunding assistance for record label / boo...,10221579,1


In [9]:
print("ep_base new shape: ", ep_base.shape)
print("ep_event new shape: ", ep_event.shape)
print("ep_job new shape: ", ep_job.shape)

print(ep_base.columns)
print(ep_event.columns)
print(ep_job.columns)

ep_base new shape:  (56089, 8)
ep_event new shape:  (11814, 7)
ep_job new shape:  (886840, 8)
Index(['name', 'employer_id', 'employer_type', 'employer_creation_date',
       'email', 'website', 'employer_size_min', 'employer_size_max'],
      dtype='object')
Index(['events_name', 'event_date', 'events_id', 'employers_name',
       'employer_id', 'event_medium', 'student_attendees_count'],
      dtype='object')
Index(['employer_name', 'employer_id', 'min_salary', 'max_salary',
       'job_creation_date', 'jobs_title', 'jobs_id', 'applications_count'],
      dtype='object')


In [10]:
# final joining
# this is temporary and subject to change
# we can have many different versions of the joined dataset, this is just one

# left join ep_event onto the ep_base
# if an employer has 3 events, that employer’s row is duplicated 3 times
# if an employer has 0 events, the event columns become NaN
ep_base_event = ep_base.merge(ep_event, on="employer_id", how="left", suffixes=("", "_event"))

# left join ep_job onto the ep_base_event
# if an employer has 5 jobs, each employer‑event row is duplicated 5 times
ep_full = ep_base_event.merge( ep_job, on="employer_id", how="left", suffixes=("", "_job") )

# final dataset = employer × event × job
# one to many relationship
# meaning employer_id with many events_id and many jobs_id will expand into multiple rows

# every employer appears at least once
# left join keeps all employers, even if they have zero events and zero jobs

print(ep_full.shape)
print(ep_full.columns)
ep_full.head(20)

(2717834, 21)
Index(['name', 'employer_id', 'employer_type', 'employer_creation_date',
       'email', 'website', 'employer_size_min', 'employer_size_max',
       'events_name', 'event_date', 'events_id', 'employers_name',
       'event_medium', 'student_attendees_count', 'employer_name',
       'min_salary', 'max_salary', 'job_creation_date', 'jobs_title',
       'jobs_id', 'applications_count'],
      dtype='object')


,name,employer_id,employer_type,employer_creation_date,email,website,employer_size_min,employer_size_max,events_name,event_date,...,employers_name,event_medium,student_attendees_count,employer_name,min_salary,max_salary,job_creation_date,jobs_title,jobs_id,applications_count
0,"lawhon & associates, inc.",392601,environmental services,2019-02-05,kblitzer@lawhon-assoc.com,http://www.lawhon-assoc.com,10.0,50.0,NaN,NaT,...,NaN,NaN,NaN,"lawhon & associates, inc.",20.0,23.0,2025-02-06,environmental technician,9660425.0,0.0
1,"lawhon & associates, inc.",392601,environmental services,2019-02-05,kblitzer@lawhon-assoc.com,http://www.lawhon-assoc.com,10.0,50.0,NaN,NaT,...,NaN,NaN,NaN,"lawhon & associates, inc.",0.0,0.0,2024-01-04,entry-level ecologist,8546132.0,0.0
2,"lawhon & associates, inc.",392601,environmental services,2019-02-05,kblitzer@lawhon-assoc.com,http://www.lawhon-assoc.com,10.0,50.0,NaN,NaT,...,NaN,NaN,NaN,"lawhon & associates, inc.",16.0,16.0,2024-05-08,cultural resource intern,8959118.0,1.0
3,"lawhon & associates, inc.",392601,environmental services,2019-02-05,kblitzer@lawhon-assoc.com,http://www.lawhon-assoc.com,10.0,50.0,NaN,NaT,...,NaN,NaN,NaN,"lawhon & associates, inc.",0.0,0.0,2023-11-02,environmental technician,8397637.0,0.0
4,"lawhon & associates, inc.",392601,environmental services,2019-02-05,kblitzer@lawhon-assoc.com,http://www.lawhon-assoc.com,10.0,50.0,NaN,NaT,...,NaN,NaN,NaN,"lawhon & associates, inc.",45000.0,50000.0,2023-10-30,architectural historian,8387960.0,0.0
5,modulus global inc.,385582,management consulting,2019-01-18,jeff@modulus.io,https://www.modulusglobal.com/,1.0,10.0,NaN,NaT,...,NaN,NaN,NaN,modulus global inc.,15.0,20.0,2023-12-30,paid public relations intern (writing / research),8536729.0,0.0
6,chameleon coffee,1008255,food & beverage,2025-01-13,careers@chameleoncoffee.com,https://www.chameleoncoffee.com,100.0,250.0,NaN,NaT,...,NaN,NaN,NaN,chameleon coffee,70000.0,80000.0,2025-01-14,traveling road brew tour brand ambassador & co...,9587228.0,3.0
7,brookstone management concepts,422901,"advertising, pr & marketing",2019-04-18,hr@brookstonemgmtconcepts.com,https://brookstonemgmtconcepts.com/,50.0,100.0,NaN,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN
8,department 13,252845,electronic & computer hardware,2018-02-21,hr@department13.com,http://department13.com,10.0,50.0,NaN,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN
9,cabrini university,37935,higher education,2015-09-04,jobs@cabrini.edu,http://www.cabrini.edu,NaN,NaN,NaN,NaT,...,NaN,NaN,NaN,cabrini university,10000.0,10000.0,2022-02-18,graduate assistant residence life,6048766.0,0.0


In [11]:
# saving datasets

#ep_base.to_csv('/content/ep_base_cleaned.csv', index=False)
#ep_event.to_csv('/content/ep_event_cleaned.csv', index=False)
#ep_job.to_csv('/content/ep_job_cleaned.csv', index=False)
#ep_full.to_csv('/content/ep_joined.csv', index=False)

# EDA